# Variant Prioritization and Interpretation


## Return context to every variant

The source paper starts us with 54 classified variant rows. GTEx, HuBMAP, and
Pharos describe the 25 genes connected to those rows. The final step joins that
gene-level context back to every published variant.

Each layer keeps its own meaning:

| Evidence | Question answered |
|---|---|
| Paper fields | What variant, phenotype, score, and class did the study report? |
| GTEx | Is the gene expressed in the selected heart tissues? |
| HuBMAP | Was expression indexed in the selected heart cell type? |
| Pharos | What protein and target-development information is available? |

We will not add these fields into one pathogenicity score. Instead, we will
state a follow-up question and select the columns that answer it.

## Build the Evidence Matrix

Run the complete integration workflow from the main notebook.

First, load the published variant table and the shared API helpers.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

REPO_ROOT = Path.cwd() if Path("api_helpers.py").exists() else Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from api_helpers import (
    fetch_gtex_context,
    fetch_hubmap_ventricular_context,
    fetch_pharos_context,
)

DATA_DIR = REPO_ROOT / "data"
variants = pd.read_csv(DATA_DIR / "variants.csv")


Next, query all three APIs for the paper's 25 genes. HuBMAP takes the longest
because it checks each gene separately. The commented lines load the saved
responses if a live service is unavailable.


In [ ]:
gene_symbols = sorted(variants["gene_symbol"].unique())
gtex = fetch_gtex_context(gene_symbols)
hubmap = fetch_hubmap_ventricular_context(gene_symbols)
pharos = fetch_pharos_context(gene_symbols)

# Backup: use the frozen 2026-08-11 responses instead of the live APIs.
# gtex = pd.read_csv(DATA_DIR / "gtex_expression.csv")
# hubmap = pd.read_csv(DATA_DIR / "hubmap_cell_expression.csv")
# pharos = pd.read_csv(DATA_DIR / "pharos_target_context.csv")


Then, create one context row per gene. This step uses the two GTEx tissues, one
HuBMAP cell type, and selected Pharos fields.


In [ ]:
gtex_wide = gtex.pivot(
    index="gene_symbol",
    columns="tissue_id",
    values="median_tpm",
).rename(
    columns={
        "Heart_Atrial_Appendage": "gtex_atrial_tpm",
        "Heart_Left_Ventricle": "gtex_ventricle_tpm",
    }
)

ventricular = (
    hubmap[hubmap["cell_type_id"] == "CL:0002131"]
    .loc[
        :,
        [
            "gene_symbol",
            "mean_normalized_expression",
            "percent_detected",
            "availability",
        ],
    ]
    .rename(
        columns={
            "mean_normalized_expression": "hubmap_ventricular_mean",
            "percent_detected": "hubmap_ventricular_percent_detected",
            "availability": "hubmap_availability",
        }
    )
)

gene_context = (
    gtex_wide.reset_index()
    .merge(ventricular, on="gene_symbol", how="left", validate="one_to_one")
    .merge(
        pharos.loc[:, ["gene_symbol", "tdl", "drug_count"]],
        on="gene_symbol",
        how="left",
        validate="one_to_one",
    )
)
assert len(gene_context) == 25


Join the 25 gene rows to all 54 variant rows. The `many_to_one` check allows
several variants to share one gene while protecting against duplicate context
rows.


In [ ]:
evidence_matrix = variants.merge(
    gene_context,
    on="gene_symbol",
    how="left",
    validate="many_to_one",
)
assert len(evidence_matrix) == len(variants) == 54
evidence_matrix.loc[
    :,
    [
        "subject_id",
        "gene_symbol",
        "hgvs_c",
        "study_class",
        "phenotype",
        "gtex_ventricle_tpm",
        "hubmap_availability",
        "tdl",
    ],
].head(10)


Ask a focused question: which DCM variant rows have measured ventricular
cardiac-myocyte context, ordered by GTEx left-ventricle expression?


In [ ]:
dcm_follow_up = (
    evidence_matrix[
        (evidence_matrix["phenotype"] == "DCM")
        & (evidence_matrix["hubmap_availability"] == "available")
    ]
    .sort_values(
        ["gtex_ventricle_tpm", "gene_symbol", "subject_id"],
        ascending=[False, True, True],
    )
    .loc[
        :,
        [
            "subject_id",
            "gene_symbol",
            "hgvs_c",
            "study_class",
            "gtex_ventricle_tpm",
            "hubmap_ventricular_percent_detected",
            "tdl",
        ],
    ]
)
dcm_follow_up


Keep HuBMAP coverage gaps visible as a unique gene list for another atlas or
experiment.


In [ ]:
coverage_gaps = (
    evidence_matrix[
        evidence_matrix["hubmap_availability"]
        != "available"
    ]
    .loc[:, ["gene_symbol", "hubmap_availability"]]
    .drop_duplicates()
    .sort_values("gene_symbol")
    .reset_index(drop=True)
)
coverage_gaps


Finally, find exact variant observations that occur in more than one subject.


In [ ]:
recurrent_variants = (
    variants.groupby(
        ["gene_symbol", "hgvs_c", "hgvs_p"],
        dropna=False,
    )
    .agg(
        subjects=("subject_id", lambda values: ", ".join(map(str, values))),
        observations=("subject_id", "size"),
    )
    .reset_index()
    .query("observations > 1")
    .sort_values(["observations", "gene_symbol"], ascending=[False, True])
)
recurrent_variants


## Write a short interpretation

Choose one row from `dcm_follow_up` and write three sentences:

1. Report the exact variant, phenotype, and study class.
2. Describe its GTEx and HuBMAP gene context with one limitation.
3. Describe its Pharos target context and name a useful next study.

Do not say that GTEx, HuBMAP, or Pharos reclassified the variant. They did not.
